In [1]:
import numpy as np
import pandas as pd

from approximate import charge_balance, parse_formula
from approximate.feature_engineering import MaterialFeatureExtractor

In [2]:
def select_feature_cols(df, feat_cols):
    """Keep numeric feature columns only; report anything skipped."""
    numeric = df[feat_cols].select_dtypes(include="number").columns.tolist()
    skipped = [c for c in feat_cols if c not in numeric]
    if skipped:
        print("Non-numeric features skipped:", skipped)
    return numeric

def find_near_constant(X, max_dominant=0.999, decimals=9):
    """Columns where one value covers more than `max_dominant` of rows.

    Returns a Series {col: dominant_fraction}, sorted descending.
    Rounding to `decimals` stops floating-point noise from counting as variation.
    """
    dominant = X.apply(
        lambda col: col.round(decimals).value_counts(normalize=True).iloc[0]
    )
    return dominant[dominant > max_dominant].sort_values(ascending=False)

def find_duplicates(X):
    """Exact duplicate columns. Returns {dropped_col: kept_col}."""
    kept, duplicates = [], {}
    for col in X.columns:
        match = next((k for k in kept if X[col].equals(X[k])), None)
        if match is None:
            kept.append(col)
        else:
            duplicates[col] = match
    return duplicates

def find_correlated(X, threshold=0.999, priority=None):
    """Greedy |Pearson r| pruning. Returns {dropped_col: (kept_col, r)}.

    Columns are visited in `priority` order first (any you list), then in
    their existing order, so the features you would rather keep win ties.
    """
    priority = [c for c in (priority or []) if c in X.columns]
    order = priority + [c for c in X.columns if c not in priority]
    corr = X[order].corr().abs()

    kept, correlated = [], {}
    for col in order:
        if kept:
            r = corr.loc[col, kept]
            if r.max() > threshold:
                correlated[col] = (r.idxmax(), round(float(r.max()), 4))
                continue
        kept.append(col)
    return correlated

def clean_features(df_feat, feat_cols, formula_col="Chemical", max_dominant=0.99, corr_threshold=0.99, priority=None):
    """Run the finders in order on unique compositions.

    Order: NaN guard -> near-constant -> duplicates -> correlated.
    Returns (drop_cols, report). Apply drop_cols to the full df_feat yourself.
    """
    feat_cols = select_feature_cols(df_feat, feat_cols)

    stats_df = df_feat
    X = stats_df.drop_duplicates(subset=formula_col)[feat_cols]

    if X.isna().any().any():
        bad = X.columns[X.isna().any()].tolist()
        raise ValueError(f"NaNs present - add a missing-value step first: {bad}")

    near_constant = find_near_constant(X, max_dominant)
    X = X.drop(columns=near_constant.index)

    duplicates = find_duplicates(X)
    X = X.drop(columns=list(duplicates))

    correlated = find_correlated(X, corr_threshold, priority)
    X = X.drop(columns=list(correlated))

    drop_cols = list(near_constant.index) + list(duplicates) + list(correlated)
    report = {
        "n_rows_stats": len(X),
        "n_start": len(feat_cols),
        "n_end": X.shape[1],
        "kept": X.columns.tolist(),
        "near_constant": near_constant,                      # col -> fraction
        "duplicates": pd.Series(duplicates, dtype=object),   # dropped -> kept
        "correlated": pd.DataFrame.from_dict(                # dropped -> (kept, r)
            correlated, orient="index", columns=["kept", "abs_r"]
        ),
    }

    return drop_cols, report

# Clean Dataset

some compositions are not compatible with the featurisation method

In [3]:
df = pd.read_csv("adj_exp_dataset.csv")
df

,UID,Chemical,Time,Temperature,P2O3ratio,Cat_Success,Impurity_Percentage
0,A00006,NaCu0.275073Mn0.724927O2,12,950,1.000000,Successful,43.56
1,A00008,NaFe0.497381Mn0.502619O2,12,950,0.997606,Successful,63.63
2,A00010,NaMn0.736534Ni0.263466O2,12,950,0.978199,Successful,31.24
3,A00011,NaCu0.250091Fe0.523806Mn0.226103O2,12,950,0.627143,Successful,41.06
4,A00013,NaCu0.3296Mn0.63802Ni0.03238O2,12,950,0.980703,Successful,56.44
...,...,...,...,...,...,...,...
552,U00365,Na0.5Cu0.388889Ni0.388889Fe0.111111Mn0.111111O2,3,950,-1.000000,Unsuccessful,88.61
553,U00366,Na0.5Fe0.388889Mn0.388889Cu0.111111Ni0.111111O2,3,950,0.263612,Successful,16.42
554,U00367,Na0.5Cu0.388889Fe0.388889Mn0.111111Ni0.111111O2,3,950,0.140791,Successful,49.99
555,U00368,Na0.5Mn0.388889Ni0.388889Fe0.111111Cu0.111111O2,3,950,0.730366,Successful,44.87


In [4]:
for formula in df["Chemical"]:
    try:
        if charge_balance(formula) is None:
            print(f"Failed: {formula} | unknown element")
    except Exception as e:
        print(f"Failed: {formula} | [{type(e).__name__}]: {e}")

In [5]:
def charge_balance_ok(formula):
    try:
        return charge_balance(formula) is not None
    except Exception:
        return False

mask = df["Chemical"].apply(charge_balance_ok)
print(f"Kept {mask.sum()} / {len(mask)} formulas")
df = df[mask].reset_index(drop=True)
df

Kept 557 / 557 formulas


,UID,Chemical,Time,Temperature,P2O3ratio,Cat_Success,Impurity_Percentage
0,A00006,NaCu0.275073Mn0.724927O2,12,950,1.000000,Successful,43.56
1,A00008,NaFe0.497381Mn0.502619O2,12,950,0.997606,Successful,63.63
2,A00010,NaMn0.736534Ni0.263466O2,12,950,0.978199,Successful,31.24
3,A00011,NaCu0.250091Fe0.523806Mn0.226103O2,12,950,0.627143,Successful,41.06
4,A00013,NaCu0.3296Mn0.63802Ni0.03238O2,12,950,0.980703,Successful,56.44
...,...,...,...,...,...,...,...
552,U00365,Na0.5Cu0.388889Ni0.388889Fe0.111111Mn0.111111O2,3,950,-1.000000,Unsuccessful,88.61
553,U00366,Na0.5Fe0.388889Mn0.388889Cu0.111111Ni0.111111O2,3,950,0.263612,Successful,16.42
554,U00367,Na0.5Cu0.388889Fe0.388889Mn0.111111Ni0.111111O2,3,950,0.140791,Successful,49.99
555,U00368,Na0.5Mn0.388889Ni0.388889Fe0.111111Cu0.111111O2,3,950,0.730366,Successful,44.87


# Featurise the dataset using approximate

In [6]:
def featurize_df(df, extractor, formula_col="Chemical"):
    """Featurise df[formula_col]. Returns (joined_df, feature_names)."""
    feat_df = pd.DataFrame(extractor.featurize_many(df[formula_col].tolist()), index=df.index)
    feature_names = [c for c in feat_df.columns if c != "formula"]

    valid_rows = feat_df[feature_names].notna().any(axis=1)
    for formula in df.loc[~valid_rows, formula_col]:
        print("Formula fail:", formula)

    return df.loc[valid_rows].join(feat_df.loc[valid_rows, feature_names]), feature_names


extractor = MaterialFeatureExtractor(
    mode="both",
    ionic_radius_unit="pm",
    enable_tm_potential=True,
    tm_cation="Na",
    tm_anion="O",
)

df_feat, feat_cols = featurize_df(df, extractor)

In [7]:
drop_cols, report = clean_features(df_feat, feat_cols)
feat_cols_clean = [c for c in feat_cols if c not in drop_cols]

n_const = int((report["near_constant"] == 1.0).sum())
print(f"Stats on {report['n_rows_stats']} unique compositions ({len(df_feat)} rows)")
print(f"Features: {report['n_start']} -> {report['n_end']}")
print(f"  constant       {n_const}")
print(f"  near-constant  {len(report['near_constant']) - n_const}")
print(f"  duplicate      {len(report['duplicates'])}")
print(f"  correlated     {len(report['correlated'])}")
report["correlated"]

Stats on 311 unique compositions (557 rows)
Features: 453 -> 136
  constant       164
  near-constant  0
  duplicate      36
  correlated     117


,kept,abs_r
all_unfilled_valence_s_sum,all_valence_s_avg,0.9992
all_unfilled_valence_s_avg,all_valence_s_avg,1.0000
all_unfilled_valence_s_dev,all_valence_s_dev,1.0000
all_unfilled_valence_s_mode,all_valence_s_mode,1.0000
all_valence_p_avg,all_valence_s_avg,1.0000
...,...,...
var_pauling_en_avg,var_pauling_en_sum,0.9999
var_gordy_en_avg,var_gordy_en_sum,0.9999
var_mb_en_sum,all_mb_en_sum,1.0000
var_mb_en_avg,all_mb_en_sum,1.0000


In [8]:
def add_parent_uid(df, uid_col="UID", parent_col="parent_uid"):
    """Add a column linking every row to its U00 parent.
 
    A00xxxx -> U00xxxx (same number); U00 rows point to themselves.
    Used later to keep each A00 row in the same fold as its parent.
    """
    df = df.copy()
    uids = df[uid_col].astype(str)
    df[parent_col] = np.where(uids.str.startswith("A00"), "U" + uids.str[1:], uids)
    return df
 
def check_orphans(df, uid_col="UID", parent_col="parent_uid"):
    """A00 rows whose U00 parent is missing (e.g. dropped by charge balance)."""
    uids = df[uid_col].astype(str)
    u00_ids = set(uids[uids.str.startswith("U00")])
    is_a00 = uids.str.startswith("A00")
    orphans = df[is_a00 & ~df[parent_col].isin(u00_ids)]
    print(f"{len(orphans)} orphaned A00 rows")
    return orphans

df_feat = add_parent_uid(df_feat)
orphans = check_orphans(df_feat)
orphans[["UID", "parent_uid", "Chemical"]]

0 orphaned A00 rows


,UID,parent_uid,Chemical


In [9]:
ELEMENTS = ["Na", "Cu", "Fe", "Mn", "Ni"]

def add_element_amounts(df, formula_col="Chemical", elements=ELEMENTS):
    """Add <El>_amount columns parsed from the formula (0 if the element is absent)."""
    df = df.copy()
    rows = [{f"{el}_amount": parse_formula(f).get(el, 0.0) for el in elements}
            for f in df[formula_col]]
    return df.join(pd.DataFrame(rows, index=df.index))

df_feat = add_element_amounts(df_feat)          # df_feat, not df
df_feat[["Chemical"] + [f"{el}_amount" for el in ELEMENTS]].head()

,Chemical,Na_amount,Cu_amount,Fe_amount,Mn_amount,Ni_amount
0,NaCu0.275073Mn0.724927O2,1.0,0.275073,0.000000,0.724927,0.000000
1,NaFe0.497381Mn0.502619O2,1.0,0.000000,0.497381,0.502619,0.000000
2,NaMn0.736534Ni0.263466O2,1.0,0.000000,0.000000,0.736534,0.263466
3,NaCu0.250091Fe0.523806Mn0.226103O2,1.0,0.250091,0.523806,0.226103,0.000000
4,NaCu0.3296Mn0.63802Ni0.03238O2,1.0,0.329600,0.000000,0.638020,0.032380


In [10]:
# ---- Save ----
import json

core_cols = ["Time"] + [f"{el}_amount" for el in ELEMENTS]
missing = [c for c in core_cols if c not in df_feat.columns]
assert not missing, f"Not in dataframe: {missing}"

feat_cols_final = core_cols + [c for c in feat_cols_clean if c not in core_cols]
meta_cols = [c for c in df_feat.columns if c not in feat_cols + core_cols]
df_out = df_feat[meta_cols + feat_cols_final]

print(f"Saving {len(df_out)} rows | {len(meta_cols)} meta cols | {len(feat_cols_final)} features")
df_out.to_csv("dataset_splits/featurised_clean.csv", index=False)
with open("dataset_splits/feature_cols.json", "w") as f:
    json.dump(feat_cols_final, f, indent=2)

Saving 557 rows | 7 meta cols | 142 features
